# EAP-IG on the Full Model

This notebook prepares the animacy dataset for EAP, runs EAP-IG on the full GPT-2 graph,
and saves ranked edges and induced nodes in separate steps.


In [ ]:
import hashlib
import importlib
import sys
from pathlib import Path
import datetime as dt

import pandas as pd
import torch
from torch.utils.data import DataLoader


def _find_animacy_root(start: Path) -> Path:
    resolved = start.resolve()
    for base in (resolved, *resolved.parents):
        for candidate in (base, base / "animacy-circuit"):
            if (
                (candidate / "dataset").is_dir()
                and (candidate / "results").is_dir()
                and (candidate / "scripts").is_dir()
            ):
                return candidate
    raise FileNotFoundError("Could not locate the animacy-circuit project root.")


project_root = _find_animacy_root(Path.cwd())
executable_dir = project_root / "scripts" / "executable"
if str(executable_dir) not in sys.path:
    sys.path.insert(0, str(executable_dir))

import circuit_finder_core as cfc
cfc = importlib.reload(cfc)

print(f"Project root: {project_root}")


In [ ]:
MODEL_NAME = "gpt2"
SEED = 42
DATASET_SAMPLE_SIZE = 500
FILTER_BATCH_SIZE = 50
ATTRIBUTION_BATCH_SIZE = 8
EVALUATION_BATCH_SIZE = 32
IG_STEPS = 5
RUN_GREEDY_BUDGET_SWEEP = False
GREEDY_COLLAPSED_EDGE_BUDGETS = [30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1200, 1400, 1600, 1800, 2000]
RUN_DAY = dt.datetime.now().strftime("%Y-%m-%d")

OUTPUT_DIR = cfc.ensure_dir(project_root / "results" / RUN_DAY / "eap_ig_full_model")


In [ ]:
model = cfc.load_model(MODEL_NAME)
tokenizer = model.tokenizer

animate_words, inanimate_words = cfc.load_animacy_targets(project_root)
animate_ids_tensor = cfc.create_target_tensor(
    animate_words,
    tokenizer,
    model.cfg.device,
)
inanimate_ids_tensor = cfc.create_target_tensor(
    inanimate_words,
    tokenizer,
    model.cfg.device,
)

cfc.verify_target_tensors(animate_words, animate_ids_tensor, tokenizer)
cfc.verify_target_tensors(inanimate_words, inanimate_ids_tensor, tokenizer)

print(
    f"Loaded {len(animate_words)} animate targets, {len(inanimate_words)} inanimate targets, "
    f"and model {MODEL_NAME} on {model.cfg.device}."
)


In [ ]:
raw_dataset = cfc.load_animacy_dataframe(project_root)
aligned_dataset = cfc.add_sequence_lengths(raw_dataset, model)

dataset_with_metrics = cfc.compute_sequence_metrics(
    aligned_dataset.copy(),
    model=model,
    tokenizer=tokenizer,
    animate_ids_tensor=animate_ids_tensor,
    inanimate_ids_tensor=inanimate_ids_tensor,
    batch_size=FILTER_BATCH_SIZE,
)

filtered_source_dataset = cfc.filter_model_success_examples(dataset_with_metrics)
if len(filtered_source_dataset) <= DATASET_SAMPLE_SIZE:
    raise ValueError(
        f"Requested {DATASET_SAMPLE_SIZE} discovery examples, but only "
        f"{len(filtered_source_dataset)} filtered examples are available."
    )

eap_discovery = filtered_source_dataset.sample(
    n=DATASET_SAMPLE_SIZE,
    random_state=SEED,
).copy()
eap_validation = filtered_source_dataset.drop(index=eap_discovery.index).copy()

if eap_validation.empty:
    raise ValueError("Validation set is empty after removing discovery examples.")

if not ((eap_discovery["clean_metric"] > 0) & (eap_discovery["corrupt_metric"] < 0)).all():
    raise ValueError("Discovery set contains invalid examples after filtering.")

if not ((eap_validation["clean_metric"] > 0) & (eap_validation["corrupt_metric"] < 0)).all():
    raise ValueError("Validation set contains invalid examples after filtering.")

sample_signature = hashlib.sha256(
    "\n".join(
        f"{clean} || {corrupt}"
        for clean, corrupt in zip(eap_discovery["clean_prefix"], eap_discovery["corrupt_prefix"])
    ).encode("utf-8")
).hexdigest()

print(
    f"Aligned examples: {len(aligned_dataset)} | "
    f"Model-success filtered examples: {len(filtered_source_dataset)} | "
    f"Discovery examples: {len(eap_discovery)} | "
    f"Validation examples: {len(eap_validation)}"
)
print(f"Discovery sample signature (sha256): {sample_signature}")

display(eap_discovery.head())
display(eap_validation.head())

eap_discovery = eap_discovery.reset_index(drop=True).copy()
eap_validation = eap_validation.reset_index(drop=True).copy()


In [ ]:
discovery_loader = DataLoader(
    cfc.CircuitPairDataset(eap_discovery),
    batch_size=ATTRIBUTION_BATCH_SIZE,
    shuffle=False,
)
validation_loader = DataLoader(
    cfc.CircuitPairDataset(eap_validation),
    batch_size=EVALUATION_BATCH_SIZE,
    shuffle=False,
)

attribute_metric = cfc.make_eap_normalized_recovery_metric(
    animate_ids_tensor,
    inanimate_ids_tensor,
)
faithfulness_metric = cfc.make_eap_normalized_recovery_vector_metric(
    animate_ids_tensor,
    inanimate_ids_tensor,
)
accuracy_metric = cfc.make_eap_accuracy_metric(
    animate_ids_tensor,
    inanimate_ids_tensor,
)

print(
    f"Discovery examples: {len(eap_discovery)} | Validation examples: {len(eap_validation)}"
)


## Run Full-Graph EAP-IG


In [ ]:
full_graph = cfc.build_graph(model)
scored_graph = cfc.attribute_graph(
    model=model,
    graph=full_graph,
    dataloader=discovery_loader,
    metric=attribute_metric,
    ig_steps=IG_STEPS,
)

print(
    f"Attributed the full graph with {int(scored_graph.real_edge_mask.sum().item())} expanded edges available."
)


## Save and Inspect Rankings


In [ ]:
ranked_edges = cfc.collapsed_edge_groups(scored_graph)
ranked_nodes = cfc.induced_node_ranking(ranked_edges)
edge_rankings_df = cfc.ranking_frame(ranked_edges)
node_rankings_df = cfc.ranking_frame(ranked_nodes)

edge_rankings_path = OUTPUT_DIR / f"full_model_edges_{RUN_DAY}.csv"
node_rankings_path = OUTPUT_DIR / f"full_model_nodes_{RUN_DAY}.csv"
edge_rankings_df.to_csv(edge_rankings_path, index=False)
node_rankings_df.to_csv(node_rankings_path, index=False)

print(f"Saved edge rankings to: {edge_rankings_path}")
print(f"Saved node rankings to: {node_rankings_path}")
display(edge_rankings_df.head(50))
display(node_rankings_df.head(50))


## Optional Greedy Budget Sweep


In [ ]:
if RUN_GREEDY_BUDGET_SWEEP:
    budget_grid = [
        budget for budget in GREEDY_COLLAPSED_EDGE_BUDGETS
        if budget <= len(ranked_edges)
    ]
    if not budget_grid:
        budget_grid = [len(ranked_edges)]

    from tqdm.auto import tqdm

    budget_rows = []
    for budget in tqdm(budget_grid, desc="Greedy budget sweep"):
        candidate_graph = cfc.build_budget_circuit(scored_graph, ranked_edges, budget)
        faithfulness_values, accuracy_values = cfc.evaluate_graph(
            model,
            candidate_graph,
            validation_loader,
            [faithfulness_metric, accuracy_metric],
            quiet=True,
            intervention="patching",
            skip_clean=False,
        )
        budget_rows.append(
            {
                "collapsed_edge_budget": int(budget),
                "expanded_edge_count": int(candidate_graph.count_included_edges()),
                "induced_node_count": int(candidate_graph.count_included_nodes() - 2),
                "faithfulness_mean": float(faithfulness_values.mean().item()),
                "faithfulness_std": float(faithfulness_values.std(unbiased=False).item()) if len(faithfulness_values) > 1 else 0.0,
                "accuracy_mean": float(accuracy_values.mean().item()),
                "accuracy_std": float(accuracy_values.std(unbiased=False).item()) if len(accuracy_values) > 1 else 0.0,
                "validation_examples": int(len(faithfulness_values)),
            }
        )

    budget_metrics_df = pd.DataFrame(budget_rows)
    budget_metrics_path = OUTPUT_DIR / f"full_model_budget_sweep_{RUN_DAY}.csv"
    budget_metrics_df.to_csv(budget_metrics_path, index=False)
    print(f"Saved budget sweep to: {budget_metrics_path}")
    display(budget_metrics_df)
else:
    print("Skipping greedy budget sweep. Set RUN_GREEDY_BUDGET_SWEEP = True to enable it.")


## Circuit Visualizations

These plots visualize the collapsed edge circuit found by EAP-IG and the validation performance across greedy edge budgets.


In [ ]:
import re
from collections import defaultdict

import plotly.graph_objects as go
from plotly.subplots import make_subplots

TOP_COLLAPSED_EDGES_TO_PLOT = min(40, len(edge_rankings_df))


def _node_metadata(node_name):
    if node_name == "input":
        return {"kind": "input", "layer": -1, "head": None}
    if node_name == "logits":
        return {"kind": "logits", "layer": -1, "head": None}

    mlp_match = re.fullmatch(r"m(\d+)", node_name)
    if mlp_match:
        return {"kind": "mlp", "layer": int(mlp_match.group(1)), "head": None}

    attn_match = re.fullmatch(r"a(\d+)\.h(\d+)", node_name)
    if attn_match:
        return {
            "kind": "attn",
            "layer": int(attn_match.group(1)),
            "head": int(attn_match.group(2)),
        }

    return {"kind": "other", "layer": 0, "head": None}


def _node_sort_key(node_name):
    meta = _node_metadata(node_name)
    type_order = {"input": 0, "attn": 1, "mlp": 2, "logits": 3, "other": 4}[meta["kind"]]
    head_order = -1 if meta["head"] is None else meta["head"]
    return (meta["layer"], type_order, head_order, node_name)


def build_layered_circuit_figure(edge_frame, top_k=40):
    if edge_frame.empty:
        raise ValueError("edge_rankings_df is empty. Run the EAP attribution cell before visualizing the circuit.")

    top_edges = edge_frame.nlargest(top_k, "abs_score").copy()
    nodes = sorted(set(top_edges["parent"]).union(top_edges["child"]), key=_node_sort_key)
    metadata = {node: _node_metadata(node) for node in nodes}

    layer_values = [meta["layer"] for meta in metadata.values() if meta["kind"] in {"attn", "mlp"}]
    max_layer = max(layer_values) if layer_values else 0

    incident_strength = defaultdict(float)
    for row in top_edges.itertuples(index=False):
        incident_strength[row.parent] += float(row.abs_score)
        incident_strength[row.child] += float(row.abs_score)

    buckets = defaultdict(list)
    for node in nodes:
        node_meta = metadata[node]
        bucket_key = node_meta["kind"] if node_meta["kind"] in {"input", "logits"} else node_meta["layer"]
        buckets[bucket_key].append(node)

    positions = {}
    for bucket_nodes in buckets.values():
        bucket_nodes.sort(key=_node_sort_key)
        center = (len(bucket_nodes) - 1) / 2
        for idx, node in enumerate(bucket_nodes):
            node_meta = metadata[node]
            if node_meta["kind"] == "input":
                x = 0.0
            elif node_meta["kind"] == "logits":
                x = float(max_layer + 2)
            elif node_meta["kind"] == "attn":
                x = float(node_meta["layer"] + 1) - 0.18
            elif node_meta["kind"] == "mlp":
                x = float(node_meta["layer"] + 1) + 0.18
            else:
                x = float(node_meta["layer"] + 1)
            positions[node] = (x, float(center - idx))

    color_map = {
        "input": "#7f7f7f",
        "attn": "#1f77b4",
        "mlp": "#ff7f0e",
        "logits": "#2f2f2f",
        "other": "#9467bd",
    }
    max_incident = max(incident_strength.values()) if incident_strength else 1.0
    max_edge_score = float(top_edges["abs_score"].max()) if len(top_edges) else 1.0

    fig = go.Figure()
    for row in top_edges.sort_values("abs_score", ascending=False).itertuples(index=False):
        x0, y0 = positions[row.parent]
        x1, y1 = positions[row.child]
        edge_color = "rgba(31,119,180,0.70)" if float(row.signed_sum) >= 0 else "rgba(214,39,40,0.70)"
        edge_width = 1.5 + 8.0 * (float(row.abs_score) / max_edge_score if max_edge_score else 0.0)
        hover_text = (
            f"Rank {int(row.rank)}<br>"
            f"{row.parent} → {row.child}<br>"
            f"Signed score: {float(row.signed_sum):.4f}<br>"
            f"|score|: {float(row.abs_score):.4f}<br>"
            f"Underlying edges: {int(row.underlying_edge_count)}"
        )
        fig.add_trace(
            go.Scatter(
                x=[x0, x1],
                y=[y0, y1],
                mode="lines",
                line=dict(color=edge_color, width=edge_width),
                hovertemplate=hover_text + "<extra></extra>",
                showlegend=False,
            )
        )

    node_x = []
    node_y = []
    node_text = []
    node_color = []
    node_size = []
    node_hover = []
    for node in nodes:
        node_meta = metadata[node]
        x, y = positions[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(node)
        node_color.append(color_map.get(node_meta["kind"], color_map["other"]))
        node_size.append(14 + 18 * (incident_strength[node] / max_incident if max_incident else 0.0))
        node_hover.append(
            "<br>".join(
                [
                    f"Node: {node}",
                    f"Kind: {node_meta['kind']}",
                    f"Layer: {node_meta['layer'] if node_meta['kind'] not in {'input', 'logits'} else 'N/A'}",
                    f"Incident |score| sum: {incident_strength[node]:.4f}",
                ]
            )
        )

    fig.add_trace(
        go.Scatter(
            x=node_x,
            y=node_y,
            mode="markers+text",
            text=node_text,
            textposition="top center",
            marker=dict(size=node_size, color=node_color, line=dict(color="white", width=1.2)),
            hovertemplate="%{customdata}<extra></extra>",
            customdata=node_hover,
            showlegend=False,
        )
    )

    tickvals = [0.0] + [float(layer + 1) for layer in range(max_layer + 1)] + [float(max_layer + 2)]
    ticktext = ["input"] + [f"L{layer}" for layer in range(max_layer + 1)] + ["logits"]

    fig.update_layout(
        title=f"Layered circuit graph for top {len(top_edges)} collapsed EAP edges",
        template="plotly_white",
        height=max(550, 32 * len(nodes)),
        hovermode="closest",
        margin=dict(l=40, r=40, t=80, b=40),
    )
    fig.update_xaxes(title="Model depth", tickvals=tickvals, ticktext=ticktext, showgrid=True, zeroline=False)
    fig.update_yaxes(title="Components within layer", showgrid=False, zeroline=False, showticklabels=False)
    fig.add_annotation(
        x=0.5,
        y=1.08,
        xref="paper",
        yref="paper",
        showarrow=False,
        text="Attention heads are offset left within each layer; MLPs are offset right. Blue edges increase the metric, red edges decrease it.",
    )
    return fig


def build_budget_curve_figure(budget_frame):
    if budget_frame.empty:
        raise ValueError("budget_metrics_df is empty. Run the greedy budget sweep before plotting the curve.")

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=budget_frame["collapsed_edge_budget"],
            y=budget_frame["faithfulness_mean"],
            mode="lines+markers",
            name="Faithfulness",
            line=dict(color="#1f77b4", width=3),
            marker=dict(size=7),
            error_y=dict(type="data", array=budget_frame["faithfulness_std"], visible=True),
            customdata=budget_frame[["validation_examples"]],
            hovertemplate=(
                "Budget=%{x}<br>"
                "Faithfulness=%{y:.4f}<br>"
                "Validation examples=%{customdata[0]}<extra></extra>"
            ),
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=budget_frame["collapsed_edge_budget"],
            y=budget_frame["accuracy_mean"],
            mode="lines+markers",
            name="Accuracy",
            line=dict(color="#ff7f0e", width=3, dash="dash"),
            marker=dict(size=7),
            error_y=dict(type="data", array=budget_frame["accuracy_std"], visible=True),
            customdata=budget_frame[["validation_examples"]],
            hovertemplate=(
                "Budget=%{x}<br>"
                "Accuracy=%{y:.4f}<br>"
                "Validation examples=%{customdata[0]}<extra></extra>"
            ),
        ),
        secondary_y=True,
    )
    fig.update_layout(
        title="Faithfulness and accuracy across greedy edge budgets",
        template="plotly_white",
        hovermode="x unified",
        margin=dict(l=50, r=50, t=80, b=50),
    )
    fig.update_xaxes(title="Collapsed edge budget")
    fig.update_yaxes(title="Faithfulness", secondary_y=False)
    fig.update_yaxes(title="Accuracy", secondary_y=True, range=[0.0, 1.0])
    return fig


layered_circuit_fig = build_layered_circuit_figure(
    edge_rankings_df,
    top_k=TOP_COLLAPSED_EDGES_TO_PLOT,
)
layered_circuit_path = OUTPUT_DIR / f"layered_circuit_top_{TOP_COLLAPSED_EDGES_TO_PLOT}_edges_{RUN_DAY}.html"
layered_circuit_fig.write_html(layered_circuit_path)
print(f"Saved layered circuit graph to: {layered_circuit_path}")
layered_circuit_fig.show()

if "budget_metrics_df" in globals() and not budget_metrics_df.empty:
    budget_curve_fig = build_budget_curve_figure(budget_metrics_df)
    budget_curve_path = OUTPUT_DIR / f"budget_sweep_curve_{RUN_DAY}.html"
    budget_curve_fig.write_html(budget_curve_path)
    print(f"Saved budget sweep curve to: {budget_curve_path}")
    budget_curve_fig.show()
else:
    print("Skipping budget sweep curve because budget_metrics_df is not available.")
